# Q4 - Convolutional Networks

In [ ]:
!git clone https://github.com/AvtandilSh1/ML_Assignment.git
import sys, os
sys.path.insert(0, '/kaggle/working/ML_Assignment')
os.chdir('/kaggle/working/ML_Assignment')

In [ ]:
import urllib.request, tarfile, os
datasets_dir = '/kaggle/working/ML_Assignment/cs231n/datasets'
os.makedirs(datasets_dir, exist_ok=True)
cifar_dir = os.path.join(datasets_dir, 'cifar-10-batches-py')
if not os.path.exists(cifar_dir):
    tar_path = os.path.join(datasets_dir, 'cifar.tar.gz')
    urllib.request.urlretrieve('http://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', tar_path)
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(datasets_dir)
    os.remove(tar_path)
print('CIFAR-10 ready.')

In [ ]:
import numpy as np
from cs231n.layers import *
from cs231n.fast_layers import *
from cs231n.classifiers.cnn import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient_array, eval_numerical_gradient
from cs231n.layer_utils import *
from cs231n.solver import Solver

def rel_error(x, y):
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

print('იმპორტი წარმატებულია')

## Conv Forward Naive - შემოწმება

In [ ]:
x_shape = (2, 3, 4, 4)
w_shape = (3, 3, 4, 4)
x = np.linspace(-0.1, 0.5, num=np.prod(x_shape)).reshape(x_shape)
w = np.linspace(-0.2, 0.3, num=np.prod(w_shape)).reshape(w_shape)
b = np.linspace(-0.1, 0.2, num=3)
conv_param = {'stride': 2, 'pad': 1}
out, _ = conv_forward_naive(x, w, b, conv_param)
correct_out = np.array([[[[-0.08759809, -0.10987781],[-0.18387192, -0.2109216 ]],
                          [[ 0.21027089,  0.21661097],[ 0.22847626,  0.23004637]],
                          [[ 0.50813986,  0.54309974],[ 0.64082444,  0.67101435]]],
                         [[[-0.98053589, -1.03143541],[-1.19128892, -1.24695841]],
                          [[ 0.69108355,  0.66880383],[ 0.59480972,  0.56776003]],
                          [[ 2.36270298,  2.36904306],[ 2.38090835,  2.38247847]]]])
print('conv_forward_naive difference (expect ~e-8):', rel_error(out, correct_out))

## Conv Backward Naive - Gradient Check

In [ ]:
np.random.seed(231)
x = np.random.randn(4, 3, 5, 5)
w = np.random.randn(2, 3, 3, 3)
b = np.random.randn(2,)
dout = np.random.randn(4, 2, 5, 5)
conv_param = {'stride': 1, 'pad': 1}
dx_num = eval_numerical_gradient_array(lambda x: conv_forward_naive(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_forward_naive(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_forward_naive(x, w, b, conv_param)[0], b, dout)
out, cache = conv_forward_naive(x, w, b, conv_param)
dx, dw, db = conv_backward_naive(dout, cache)
print('dx error:', rel_error(dx, dx_num))
print('dw error:', rel_error(dw, dw_num))
print('db error:', rel_error(db, db_num))

## MaxPool Forward/Backward - შემოწმება

In [ ]:
x_shape = (2, 3, 4, 4)
x = np.linspace(-0.3, 0.4, num=np.prod(x_shape)).reshape(x_shape)
pool_param = {'pool_width': 2, 'pool_height': 2, 'stride': 2}
out, _ = max_pool_forward_naive(x, pool_param)
correct_out = np.array([[[[-0.26315789,-0.24842105],[-0.20421053,-0.18947368]],[[-0.14526316,-0.13052632],[-0.08631579,-0.07157895]],[[-0.02736842,-0.01263158],[0.03157895,0.04631579]]],[[[0.09052632,0.10526316],[0.14947368,0.16421053]],[[0.20842105,0.22315789],[0.26736842,0.28210526]],[[0.32631579,0.34105263],[0.38526316,0.4]]]])
print('max_pool_forward difference (expect ~e-8):', rel_error(out, correct_out))
np.random.seed(231)
x = np.random.randn(3, 2, 8, 8)
dout = np.random.randn(3, 2, 4, 4)
dx_num = eval_numerical_gradient_array(lambda x: max_pool_forward_naive(x, pool_param)[0], x, dout)
_, cache = max_pool_forward_naive(x, pool_param)
dx = max_pool_backward_naive(dout, cache)
print('max_pool_backward dx error (expect ~e-12):', rel_error(dx, dx_num))

## ThreeLayerConvNet - Gradient Check

In [ ]:
np.random.seed(231)
X = np.random.randn(2, 3, 16, 16)
y = np.random.randint(10, size=2)
model = ThreeLayerConvNet(num_filters=3, filter_size=3, input_dim=(3,16,16), hidden_dim=7, dtype=np.float64)
loss, grads = model.loss(X, y)
for name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    num_grad = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-6)
    print(f'{name}: {rel_error(num_grad, grads[name]):.2e}')

## Spatial Batchnorm - Gradient Check

In [ ]:
np.random.seed(231)
N, C, H, W = 2, 3, 4, 5
x = 5 * np.random.randn(N, C, H, W) + 12
gamma = np.random.randn(C)
beta = np.random.randn(C)
dout = np.random.randn(N, C, H, W)
bn_param = {'mode': 'train'}
fx = lambda x: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]
_, cache = spatial_batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = spatial_batchnorm_backward(dout, cache)
dx_num = eval_numerical_gradient_array(fx, x, dout)
print('dx error:     ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(eval_numerical_gradient_array(lambda a: spatial_batchnorm_forward(x, a, beta, bn_param)[0], gamma, dout), dgamma))
print('dbeta error:  ', rel_error(eval_numerical_gradient_array(lambda b: spatial_batchnorm_forward(x, gamma, b, bn_param)[0], beta, dout), dbeta))

## Spatial GroupNorm - Gradient Check

In [ ]:
np.random.seed(231)
N, C, H, W, G = 2, 6, 4, 5, 2
x = 5 * np.random.randn(N, C, H, W) + 12
gamma = np.random.randn(1, C, 1, 1)
beta = np.random.randn(1, C, 1, 1)
dout = np.random.randn(N, C, H, W)
gn_param = {}
_, cache = spatial_groupnorm_forward(x, gamma, beta, G, gn_param)
dx, dgamma, dbeta = spatial_groupnorm_backward(dout, cache)
dx_num = eval_numerical_gradient_array(lambda x: spatial_groupnorm_forward(x, gamma, beta, G, gn_param)[0], x, dout)
print('dx error:     ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(eval_numerical_gradient_array(lambda a: spatial_groupnorm_forward(x, a, beta, G, gn_param)[0], gamma, dout), dgamma))
print('dbeta error:  ', rel_error(eval_numerical_gradient_array(lambda b: spatial_groupnorm_forward(x, gamma, b, G, gn_param)[0], beta, dout), dbeta))

## ThreeLayerConvNet - სწავლება

In [ ]:
np.random.seed(231)
data = get_CIFAR10_data()
small_data = {'X_train': data['X_train'][:100], 'y_train': data['y_train'][:100],
              'X_val': data['X_val'], 'y_val': data['y_val']}
model = ThreeLayerConvNet(weight_scale=1e-2)
solver = Solver(model, small_data, num_epochs=10, batch_size=50,
                update_rule='adam', optim_config={'learning_rate': 1e-3}, verbose=False)
solver.train()
print('train_acc:', max(solver.train_acc_history))
print('val_acc:  ', max(solver.val_acc_history))

## MLflow - DagsHub-ზე Logging

In [ ]:
!pip install -q mlflow
import mlflow, os
os.environ['MLFLOW_TRACKING_USERNAME'] = 'ashos22'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '78b25562699413e85d83414b742443e9df7c5cb5'
mlflow.set_tracking_uri('https://dagshub.com/ashos22/ML_Assignment.mlflow')
mlflow.set_experiment('Q4-ConvNet')
with mlflow.start_run(run_name='three-layer-convnet'):
    mlflow.log_params({'num_filters': 32, 'filter_size': 7, 'hidden_dim': 100, 'num_epochs': 10, 'n_train': 100})
    mlflow.log_metrics({'best_val_acc': float(max(solver.val_acc_history)), 'best_train_acc': float(max(solver.train_acc_history))})
print('Logged to DagsHub.')